# Customer Churn: Model Training, Tuning, and Ensembling
In this notebook, we will train our from-scratch implementations of Logistic Regression, Naive Bayes, and Neural Networks. We will also demonstrate the tuning process for the Neural Network and evaluate our final Ensemble model.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src is in the path
sys.path.append(os.path.abspath(os.path.join('..')))

from src.models.naive_bayes import GaussianNaiveBayes
from src.models.neural_network import MLPClassifier

# Set seed for reproducibility
np.random.seed(42)

### 1. Load and Split Data

In [ ]:
# Read the preprocessed dataset
data_path = os.path.join('..', 'data', 'processed', 'cleaned_churn_data.csv')
df = pd.read_csv(data_path)

X = df.drop(columns=['Churn']).values.astype(np.float64)
y = df['Churn'].values

print(f"Data shape: {X.shape}")

# Shuffle data
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

# Split: 60% Train, 20% Val, 20% Test
n = len(X)
train_end = int(0.6 * n)
val_end = int(0.8 * n)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

### 2. Train Naive Bayes

In [ ]:
print("Training Naive Bayes...")
nb = GaussianNaiveBayes()
nb.fit(X_train, y_train)

nb_val_preds = nb.predict(X_val)
nb_val_acc = np.mean(nb_val_preds == y_val)
print(f"Naive Bayes Validation Accuracy: {nb_val_acc:.4f}")

### 3. Train and Tune Neural Network

In [ ]:
print("Training Neural Network (Tuning Process)...")

# Model A: Learning rate too small (underfitting)
nn_underfit = MLPClassifier(hidden_layer_sizes=(16, 8), learning_rate=0.001, max_iter=500)
print("Training Model A (LR=0.001) - intentionally showing underfitting...")
nn_underfit.fit(X_train, y_train, X_val, y_val)

# Model B: Better learning rate
nn_tuned = MLPClassifier(hidden_layer_sizes=(16, 8), learning_rate=0.1, max_iter=800)
print("Training Model B (LR=0.1) - tuned hyperparameters...")
nn_tuned.fit(X_train, y_train, X_val, y_val)

nn_val_acc = nn_tuned.history['val_accuracy'][-1]
print(f"Neural Network Tuned Validation Accuracy: {nn_val_acc:.4f}")

# Plotting Learning Curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(nn_underfit.history['loss'], label='LR=0.001 (Train)')
plt.plot(nn_underfit.history['val_loss'], label='LR=0.001 (Val)', linestyle='--')
plt.plot(nn_tuned.history['loss'], label='LR=0.1 (Train)')
plt.plot(nn_tuned.history['val_loss'], label='LR=0.1 (Val)', linestyle='--')
plt.title('Neural Network Tuning: Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Cross-Entropy Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(nn_tuned.history['accuracy'], label='Train Acc')
plt.plot(nn_tuned.history['val_accuracy'], label='Val Acc', linestyle='--')
plt.title('Neural Network Tuned: Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

### 4. Final Evaluation & Ensemble (Test Set)

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.1, iters=1000):
        self.lr, self.iters = lr, iters
    def fit(self, X, y):
        m, n = X.shape
        self.w, self.b = np.zeros(n), 0
        for _ in range(self.iters):
            z = np.dot(X, self.w) + self.b
            a = 1 / (1 + np.exp(-np.clip(z, -250, 250)))
            dz = a - y
            self.w -= self.lr * ((1/m) * np.dot(X.T, dz))
            self.b -= self.lr * ((1/m) * np.sum(dz))
    def predict_proba(self, X):
        z = np.dot(X, self.w) + self.b
        prob1 = 1 / (1 + np.exp(-np.clip(z, -250, 250)))
        return np.vstack((1 - prob1, prob1)).T

lr = LogisticRegression()
lr.fit(X_train, y_train)

# Predictions
lr_probs = lr.predict_proba(X_test)
nb_probs = nb.predict_proba(X_test)
nn_probs = nn_tuned.predict_proba(X_test)

lr_acc = np.mean(np.argmax(lr_probs, axis=1) == y_test)
nb_acc = np.mean(np.argmax(nb_probs, axis=1) == y_test)
nn_acc = np.mean(np.argmax(nn_probs, axis=1) == y_test)

ensemble_probs = (lr_probs + nb_probs + nn_probs) / 3.0
ensemble_acc = np.mean(np.argmax(ensemble_probs, axis=1) == y_test)

print(f"Logistic Regression: {lr_acc:.4f}")
print(f"Naive Bayes: {nb_acc:.4f}")
print(f"Neural Network: {nn_acc:.4f}")
print(f"Ensemble (LR + NB + NN): {ensemble_acc:.4f}")

plt.figure(figsize=(8, 5))
models = ['Log Reg', 'Naive Bayes', 'Neural Network', 'Ensemble']
accuracies = [lr_acc, nb_acc, nn_acc, ensemble_acc]
plt.bar(models, accuracies, color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'])
plt.ylim(0.5, 1.0)
plt.title('Final Model Comparisons on Test Set')
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.01, f"{v:.3f}", ha='center', fontweight='bold')
plt.show()